<a href="https://colab.research.google.com/github/TOTVScontext/DataScience/blob/main/context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Leonardo da Silva Pinto | RM: 564929
#Samuel Enzo Domingues Monteiro | RM: 564391
#Guilherme de Araujo Moreira | RM: 561848

# 🚀 Sprint 3: Machine Learning Supervisionado para NLP e Classificação de Negócios
**Disciplina:** Data Science and Statistical Computing
**Objetivo:** Evoluir o pipeline de dados para classificar trechos de transcrições de reuniões em categorias relevantes de negócio, comparando algoritmos supervisionados e justificando as métricas de avaliação.

---
### 📌 1. Definição do Problema e Estratégia de Rotulagem
O problema de negócio escolhido para esta sprint é a **Classificação de Risco de Churn ou Insatisfação**.

**Por que esta escolha?**
A base de dados reais de transcrições (`ANON_transcricao.json`) não possui uma variável alvo pré-definida de satisfação. No entanto, a nossa base de dados corporativa em formato tabular (`reunioes_transcricoes_mockado.csv`) contém a coluna `NOTA_NPS` (escala 0 a 10) atrelada a cada reunião.

Utilizar o NPS como base para a rotulagem supervisionada é a estratégia mais robusta e menos enviesada para identificar risco, pois não depende de heurísticas baseadas em palavras-chave criadas por nós, mas sim de uma pesquisa de satisfação real do cliente.

**Regra de Binarização (Target):**
*   **Risco (Classe 1):** NPS de 0 a 6 (Detratores).
*   **Neutro/Seguro (Classe 0):** NPS de 7 a 10 (Neutros e Promotores).

Essa classificação atribuída à reunião será propagada para cada fala/trecho da transcrição correspondente, permitindo que o algoritmo aprenda quais padrões de linguagem indicam um cliente em zona de risco.

In [ ]:
# ==============================================================================
# 1. SETUP, IMPORTAÇÕES E CARGA DOS DADOS
# ==============================================================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Download das stopwords do NLTK (pt-BR)
nltk.download('stopwords', quiet=True)
stop_words_pt = stopwords.words('portuguese')

print("⏳ Carregando as bases de dados...")

# Leitura do CSV (Base com NPS - Usada para o treinamento de Churn)
df_csv = pd.read_csv('reunioes_transcricoes_mockado.csv')

# Leitura do JSON Lines (Base real de produção - Usada para validação de Produto)
df_json = pd.read_json('ANON_transcricao.json', lines=True)

print(f"✅ CSV carregado: {df_csv.shape[0]} reuniões.")
print(f"✅ JSON carregado: {df_json.shape[0]} reuniões reais.")

⏳ Carregando as bases de dados...
✅ CSV carregado: 500 reuniões.
✅ JSON carregado: 1174 reuniões reais.


### ✂️ 2. Segmentação e Limpeza de Texto (Feature Engineering)
O modelo não deve classificar a reunião inteira como um bloco único de texto, mas sim **trechos e diálogos**. Abaixo, aplicaremos o parsing para quebrar a coluna de transcrição em falas individuais e aplicaremos a higienização textual rigorosa (lowercasing, remoção de pontuação e stopwords).

In [ ]:
# ==============================================================================
# 2. SEGMENTAÇÃO E HIGIENIZAÇÃO DOS DADOS
# ==============================================================================
records = []

# 2.1 Quebra das transcrições do CSV em trechos/falas
for idx, row in df_csv.iterrows():
    texto = str(row['ANON_TRANSCRICAO'])
    nps = row['NOTA_NPS']

    linhas = texto.split('\n')
    for linha in linhas:
        if ':' in linha:
            locutor, fala = linha.split(':', 1)
            records.append({
                'ID_MEETING': row['ID_MEETING'],
                'LOCUTOR': locutor.strip(),
                'FALA_BRUTA': fala.strip(),
                'NOTA_NPS': nps
            })

df_trechos = pd.DataFrame(records)

# 2.2 Criação da Variável Alvo (Risco de Churn)
# NPS <= 6 é Risco (1), NPS >= 7 é Seguro/Neutro (0)
df_trechos['TARGET_RISCO'] = df_trechos['NOTA_NPS'].apply(lambda x: 1 if x <= 6 else 0)

# 2.3 Função de Limpeza de Texto (NLP Básica)
def limpar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^\w\s]', '', texto) # Remove pontuação
    palavras = texto.split()
    # Remove stopwords mas mantém eventuais marcações como [PESSOA]
    palavras = [w for w in palavras if w not in stop_words_pt]
    return ' '.join(palavras)

df_trechos['FALA_LIMPA'] = df_trechos['FALA_BRUTA'].apply(limpar_texto)

print(f"✅ Segmentação concluída. {df_csv.shape[0]} reuniões geraram {df_trechos.shape[0]} trechos de fala para análise.")

✅ Segmentação concluída. 500 reuniões geraram 3750 trechos de fala para análise.
